# Glassbox linear models

This notebook fits the same sparse linear structure with two training losses. Hyperparameters are selected on the final 20% of complete purchase dates inside the training split. The formal validation split is used only after both choices are fixed.

In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

from glassbox import run_glassbox
from preprocessing import PROCESSED, RAW, ROOT

raw = Path(os.environ.get("GLASSBOX_RAW", RAW))
processed = Path(os.environ.get("GLASSBOX_PROCESSED", PROCESSED))
result_dir = Path(os.environ.get(
    "GLASSBOX_RESULTS", ROOT / "results" / "validation" / "acquisition_prices"
))

In [ ]:
result = run_glassbox(processed=processed, raw=raw, directory=result_dir)
print(f"Inner training ends on {result['inner_training_end']:%Y-%m-%d}")
display(result["tuning"])
display(result["auc"].rename_axis("model").to_frame())

## Coefficient interpretation

A numeric coefficient is the score change from a one-training-standard-deviation increase. A categorical coefficient is relative to the displayed reference level. L2 coefficients change the continuous risk score; Logistic coefficients change log-odds, and their exponentiated values are odds ratios. Coefficient magnitudes are not compared across the two losses.

In [ ]:
focus_fields = [
    "VehicleAge", "VehOdo",
    "MMRAcquisitionAuctionAveragePrice",
    "MMRAcquisitionAuctionCleanPrice",
    "MMRAcquisitionRetailAveragePrice",
    "MMRAcquisitonRetailCleanPrice",
]
columns = [
    "encoded_feature", "reference_level", "training_scale",
    "Glassbox_L2_coefficient", "Glassbox_Logloss_coefficient",
    "Glassbox_Logloss_odds_ratio", "direction_stable",
]
display(result["coefficients"].loc[
    result["coefficients"]["original_field"].isin(focus_fields), columns
])
display(result["coefficients"].loc[
    result["coefficients"]["direction_stable"], columns
].head(20))

In [ ]:
display(Image(filename=str(result_dir / "glassbox_top_effects.png")))

Positive coefficients are associated with higher predicted bad-buy risk and negative coefficients with lower predicted risk, conditional on the other included fields. These are predictive associations rather than causal effects. The plot uses a separate coefficient scale for each loss.